# 第4回 演習：物差しを選び、係数を説明に変える（解答例・教員用）


---


## 前回からの接続


第3回は、民泊のホストとして、データを整えてから1泊の値段を付けました。整えるたびに外れ幅の平均で測りましたが、「その物差しだけでよいのか」「高い部屋を外してよかったのか」には、まだ答えていません。そして、式がどれだけ当たるかは測れても、式の中身、つまりどの条件が値段をどれだけ押し上げているのかは、まだ一度も読んでいません。

今日は、米国アイオワ州エイムズ市の住宅の売買価格のデータで、この2つに取り組みます。


---


## 今日の分析目標


あなたは住宅の査定担当者です。値段を当てる式をどの物差しで選ぶかを決め、お客さんに「この家は、なぜこの値段なのか」を条件ごとに説明する。

外れ幅の平均（MAE）に加えて、大外れを重く見る物差し（RMSE）を自分で関数にし、価格を対数に変えて当てる式と比べます。そのあと、式の係数を日本語の文に直し、1軒の家の査定書を作ります。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、米国エイムズ市で住宅の査定を担当しています。家の条件から値段を見積もり、お客さんに伝えます。そのとき、なぜその値段になったのかも説明します。

決めたいことは2つです。値段を当てる式を、どの物差しで選ぶか。そして、お客さんに「この家は、なぜこの値段なのか」をどう説明するかです。


### 手元のデータ

エイムズ市の資産評価事務所が記録した、住宅の売買2,930件です（De Cock, 2011）。1行が1軒の売買です。答えの列`SalePrice`は、その家が売れた価格（米ドル）です。


### 最後に出す答え

値段を当てる式と、それを選んだ物差しです。そして、1軒の家の査定書です。平均的な家より高い、あるいは安い理由を、条件ごとの金額で示します。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- MAEとRMSEを関数として読み書きし、大外れが2つの物差しをどう動かすかを説明できる
- 右に長く歪んだ値を対数に変えて当て、元の単位に戻して評価できる
- 1回の分け方ではなく、交差検証（cross-validation）の平均で式を比べ、結果を表にまとめられる
- 線形回帰（linear regression）の係数を、「ほかの条件をそろえたうえで」を付けた日本語の文に直せる
- 単位の違う係数はそのまま比べられないことを説明し、標準化（standardization）した係数で大きさを比べられる
- よく似た列の組（多重共線性）があると、係数を一つずつ読めなくなることを確かめられる
- 1軒の家の予測を、条件ごとの押し上げ・押し下げに分けて、f-stringで文にできる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の評価と、③ の解釈が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. どの売買を手がかりにするか


第3回の民泊でも、最初に、手がかりにする部屋を目標から決めました。今日も、査定の手がかりにする売買を先に選びます。エイムズ市の資産評価事務所が記録した、2006〜2010年の住宅の売買2,930件です（De Cock, 2011）。1行が1軒の売買で、`SalePrice`が売れた価格（米ドル）です。

列は80本もあります。売買の条件（`Sale Condition`）には、ふつうの売買（Normal）のほかに、建築中の家の売買（Partial）、差し押さえなどの特殊な売買（Abnorml）、家族間の売買（Family）などが混ざっています。第3回と同じく、目標から使う行を決めます。査定担当が説明したいのはふつうの売買の値段です。さらに、居住面積が4,000平方フィート（約372㎡）を超える家も外します。データ全体でも5軒しかない極端に広い家です（ふつうの売買の中では1軒）。ほかの家とは値段の付き方が違うため、データの作者も「授業で使うなら外すとよい」と勧めています。面積は平方フィートなので、㎡ に直しておきます（1平方フィート ≒ 0.0929㎡）。


### お手本：読み込んで、ふつうの売買に絞る

- 役割：データを読み込み、ふつうの売買で広すぎない家に絞り、面積を㎡に直す
- 入力：データの置き場所とファイル名
- 出力：表`df`


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/ames_housing.csv')
normal = raw['Sale Condition'] == 'Normal'         # ふつうの売買
small = raw['Gr Liv Area'] <= 4000                 # 広すぎない家
df = raw[normal & small].reset_index(drop=True)
for c in ['Gr Liv Area', 'Total Bsmt SF', 'Garage Area', 'Lot Area']:
    df[c] = df[c] * 0.0929                         # 平方フィート → ㎡
print(len(df), '軒')
print(df['SalePrice'].describe().round(0))


2,412軒になりました。価格の中央値は約16万ドルです。今日は、80本の列のうち、査定でよく話題になる次の11本の数の列を使います。

| 列 | 意味 | 単位 |
|---|---|---|
| `Overall Qual` | 家の全体の品質（材料と仕上げ） | 1〜10の10段階 |
| `Gr Liv Area` | 地上の居住面積 | ㎡ |
| `Total Bsmt SF` | 地下室の面積 | ㎡ |
| `Garage Cars` | 車庫に入る車の台数 | 台 |
| `Garage Area` | 車庫の面積 | ㎡ |
| `Year Built` | 建てられた年 | 年 |
| `Year Remod/Add` | 改修・増築した年（していなければ建てた年） | 年 |
| `Full Bath` | 地上のバスルームの数 | 個 |
| `TotRms AbvGrd` | 地上の部屋の数 | 個 |
| `Fireplaces` | 暖炉の数 | 個 |
| `Lot Area` | 敷地の面積 | ㎡ |


### 変えてみる：訓練とテストに分け、線形回帰を立てる


In [ ]:
feat = ['Overall Qual', 'Gr Liv Area', 'Total Bsmt SF', 'Garage Cars',
        'Garage Area', 'Year Built', 'Year Remod/Add', 'Full Bath',
        'TotRms AbvGrd', 'Fireplaces', 'Lot Area']
tr, te = train_test_split(df, test_size=0.2, random_state=42)
X_tr, X_te = tr[feat], te[feat]
y_tr, y_te = tr['SalePrice'].values, te['SalePrice'].values
m = LinearRegression().fit(X_tr, y_tr)
print(m.score(X_te, y_te))                         # テストの R²


テストのR²は0.867。よく当たる式に見えます。では、何ドルくらい外れるのでしょうか。


---


## 2. 物差しを、関数にする


テストのR²は0.867で、よく当たる式に見えます。でも査定では、何ドル外れるのかを知りたいところです。

### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが関数の名前と引数（入力）、`return`の後ろが戻り値（出力）でした（第3回）。関数は作っただけでは動かず、`名前(値)`で呼び出します。


### お手本：外れ幅の平均（MAE）

- 役割：実際の価格と予測から、外れ幅の平均（MAE、mean absolute error）を出す
- 入力：実際の値`y`と、予測`pred`
- 出力：外れ幅の平均（ドル）


In [ ]:
def mae(y, pred):          # 外れ幅の平均（MAE）
    return round(np.abs(y - pred).mean())

print(mae(y_te, np.median(y_tr)))       # 基準：どの家も中央値と答える
print(mae(y_te, m.predict(X_te)))       # 線形回帰


どの家も中央値と答える基準では50,318ドル、線形回帰では19,061ドル外れます。基準の半分以下です。

ただ、外れ幅の平均は、1万ドルの外れが10回あっても、10万ドルの外れが1回と9回の当たりでも、同じ値になります。査定で10万ドル外すのは、大きな失敗です。大外れを重く見る物差しも欲しくなります。

そこで使うのがRMSE（root mean squared error）です。外れを二乗して平均し、最後にルートを取って、ドルの単位に戻します。第2回の二乗和と同じく、二乗するので大外れが重く数えられます。


### 変えてみる：RMSEを関数にする

`mae`のお手本の中身を、「外れを二乗して平均し、ルートを取る」に変えます。ルートは`np.sqrt`です。


In [ ]:
def rmse(y, pred):        # 二乗して平均し、ルートを取る
    return round(np.sqrt(((y - pred) ** 2).mean()))

print(rmse(y_te, m.predict(X_te)))
mse = mean_squared_error(y_te, m.predict(X_te))
print(round(np.sqrt(mse)))                 # 答え合わせ


線形回帰のRMSEは25,627ドルで、MAE（19,061ドル）より大きくなりました。scikit-learnの`mean_squared_error`（外れの二乗の平均、MSE）のルートとも一致します。RMSEは、いつもMAE以上になります。その差が大きいほど、大外れが混ざっているということです。


### TODO①：大外れは、二乗の合計のどれだけを占めるか

テストの家ごとの外れの二乗`err2`を大きい順に並べ、上位5%（`len(err2) // 20`軒）だけの合計が、全体の合計の何割かを表示してください。


In [ ]:
# TODO: 外れの二乗 err2 を大きい順に並べ、

# 解答例①：上位5%が占める割合
err2 = (y_te - m.predict(X_te)) ** 2
top = np.sort(err2)[::-1][:len(err2) // 20]      # 大きい順の上位5%
print(len(top), '軒')
print(round(top.sum() / err2.sum(), 3))


外れの大きい上位5%の家（24軒）だけで、外れの二乗の合計の4割（0.417）を占めています。RMSEは、この大外れを重く見ているのです。


---


## 3. 高い家に、引っぱられていないか


上位5%の家だけで、外れの二乗の合計の4割を占めていました。その大外れは、どんな家で起きているのでしょうか。価格の分布を見てみます。


In [ ]:
# @title 図：価格の分布
plt.hist(df['SalePrice'], bins=60, color='#0066cc')
plt.xlabel('売れた価格（ドル）'); plt.ylabel('家の数'); plt.show()


右に長く歪んでいます（第3回の民泊の料金と同じ形です）。高い家は数が少なく、値段の幅も大きいので、式はそこで大きく外していないでしょうか。


### 確かめる：高い家だけの外れ幅

- 役割：30万ドルを超えるテストの家だけで、外れ幅の平均を出す
- 入力：テストの価格`y_te`と、そのまま当てた予測
- 出力：家の数、外れ幅の平均、いちばん安い予測

`y_te > 300000`は、家ごとのTrue/Falseの並びです。`y_te[high]`のように`[ ]`に入れると、Trueの家だけが取り出されます。


In [ ]:
pred_raw = m.predict(X_te)                 # そのまま当てた予測
high = y_te > 300000                       # 30万ドルを超える家
print(high.sum(), '軒')
print(mae(y_te[high], pred_raw[high]))     # 高い家の外れ幅の平均
print(pred_raw.min().round(0))             # いちばん安い予測


30万ドルを超えるテストの30軒では、外れ幅の平均が52,197ドルにもなります（全体では19,061ドル）。しかも、そのまま当てる直線の式は、ある家に−3,296ドルという、ありえないマイナスの価格を付けていました。

こういうとき、価格そのものではなく、価格の対数（`np.log`）を当てる、というやり方があります。対数にすると、10万ドルと20万ドルの差と、20万ドルと40万ドルの差が同じ幅になり、右に長い尾が縮みます。当てたあとで`np.exp`を使えば、ドルに戻せます。戻した値は、必ずプラスになります。


### お手本：対数を当てて、ドルに戻す

- 役割：価格の対数を当てる線形回帰を立て、予測を`np.exp`でドルに戻す
- 入力：`X_tr`・`y_tr`・`X_te`
- 出力：ドルに戻したテストの予測`pred_log`


In [ ]:
m_log = LinearRegression().fit(X_tr, np.log(y_tr))   # 対数を当てる
pred_log = np.exp(m_log.predict(X_te))             # ドルに戻す
print(pred_log[:3].round(0))
print(pred_log.min().round(0))                     # いちばん安い予測


### TODO②：2つの式を、2つの物差しで比べる

2節の`mae`と`rmse`を使って、そのまま当てた式（`pred_raw`）と、対数で当てた式（`pred_log`）の、MAEとRMSEを表示してください。


In [ ]:
# TODO: 2つの式の MAE と RMSE を表示する

# 解答例②：2つの式を、2つの物差しで
pred_raw = m.predict(X_te)
print('そのまま: MAE', mae(y_te, pred_raw), ' RMSE', rmse(y_te, pred_raw))
print('対数    : MAE', mae(y_te, pred_log), ' RMSE', rmse(y_te, pred_log))


MAEでは、対数の式が16,408ドルと、そのままの式（19,061ドル）よりずっと良くなりました。ところがRMSEでは、対数の式が26,296ドルと、そのままの式（25,627ドル）より悪いのです。物差しを変えると、式の順位が入れ替わりました。

では、どちらの式を信じればよいのでしょうか。その前に、もう1つ確かめることがあります。


---


## 4. 1回の分け方で、決めてよいか


TODO②の比べは、1回の分け方（`random_state=42`）の結果でした。第2回で見たとおり、1回の値は分け方の運で動きます。そこで、5分割の交差検証で、5回ぶんのMAEとRMSEを測ります。今度は`cross_val_score`を使わず、`for`文で自分で5回まわして、結果を表にまとめます。


### おさらい：for文

```python
for i in range(3):
    print(i)          # 0, 1, 2 と表示される
```

並びの中身を1つずつ名前に入れながら、字下げした行を繰り返す書き方でした（第2回）。


### 分け方を、5回ぶん出す

`KFold`は、データを5つに分ける道具です。`kf.split(df)`は、5回ぶんの（訓練の行番号の並び, テストの行番号の並び）の組を、1回ずつ順に出します。`for a, b in`と書くと、その組を`a`と`b`に分けて受け取れます。


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for a, b in kf.split(df):
    print(len(a), len(b), a[:3])     # 訓練の数・テストの数・行番号


1回ごとに、訓練は約1,930軒、テストは約480軒です。5回で、どの家もちょうど1回ずつテストに回ります。

### 番号を付けて回す：enumerate

何回目かを数えながら回したいときは、`enumerate`を使います。中身に0から番号を付けて、（番号, 中身）の組で出します。


In [ ]:
for k, w in enumerate(['A', 'B', 'C']):
    print(k, w)          # 0 A、1 B、2 C と表示される


### お手本：5回まわして、結果を表にまとめる

- 役割：データを5つに分け、各回で線形回帰を学んでテストし、MAEとRMSEを表にまとめる
- 入力：表`df`と、分け方`kf`
- 出力：5行の表`table`

1行目の`for k, (a, b) in enumerate(kf.split(df)):`は、上の2つを合わせた形です。`k`が回の番号（0から）、`(a, b)`が分け方の組です。`df.iloc[a]`は、行番号（0から数えた位置）で行を取り出す書き方で、`a`の行が訓練`t`、`b`の行がテスト`s`になります。

各回の結果を辞書（`{'名前': 値}`）にして、リスト`rows`にためます。最後に`pd.DataFrame(rows)`に渡すと、辞書1つが1行の表になります。関数`mae`・`rmse`を、`for`文の中で呼び出しているところがポイントです。


In [ ]:
rows = []
for k, (a, b) in enumerate(kf.split(df)):
    t, s = df.iloc[a], df.iloc[b]      # その回の訓練とテスト
    m1 = LinearRegression().fit(t[feat], t['SalePrice'])
    p1 = m1.predict(s[feat])
    rows.append({'回': k + 1,
                 'MAE_そのまま': mae(s['SalePrice'], p1),
                 'RMSE_そのまま': rmse(s['SalePrice'], p1)})
table = pd.DataFrame(rows)
print(table)


### TODO③：対数の式も、同じ表に並べる

お手本のセルをコピーして、`for`文の中で対数の式も学び（3節のお手本と同じ）、辞書に`'MAE_対数'`と`'RMSE_対数'`を足してください。そして、表と、列ごとの平均を表示してください。平均は、回の番号の列を外してから取ります（`table.drop(columns='回').mean()`）。


In [ ]:
# TODO: お手本の for 文の中に、対数の式の予測 p2 を足し、

# 解答例③：対数の式も並べる
rows = []
for k, (a, b) in enumerate(kf.split(df)):
    t, s = df.iloc[a], df.iloc[b]      # その回の訓練とテスト
    m1 = LinearRegression().fit(t[feat], t['SalePrice'])
    p1 = m1.predict(s[feat])
    m2 = LinearRegression().fit(t[feat], np.log(t['SalePrice']))
    p2 = np.exp(m2.predict(s[feat]))
    rows.append({'回': k + 1,
                 'MAE_そのまま': mae(s['SalePrice'], p1),
                 'RMSE_そのまま': rmse(s['SalePrice'], p1),
                 'MAE_対数': mae(s['SalePrice'], p2),
                 'RMSE_対数': rmse(s['SalePrice'], p2)})
table = pd.DataFrame(rows)
print(table)
print(table.drop(columns='回').mean().round(0))


5回の平均で見ると、MAEはそのまま19,304ドルに対して対数16,096ドル、RMSEはそのまま27,004ドルに対して対数22,703ドル。どちらの物差しでも、対数の式のほうが良いのです。TODO②でRMSEの順位が逆になったのは、1回目の分け方でたまたま起きたことでした（表の1回目の行だけ、RMSEが逆転しています。1回目のテストの483軒は、TODO②のテストと同じ家です）。

物差しを決め、交差検証の平均で比べてから、式を選ぶ。査定担当として、高い家でも大外れしにくく、マイナスの価格も出さない対数の式を、当てるための式に選びます。


---


## 5. 係数を読む：「ほかの条件をそろえたうえで」


当てる式は、対数の式に決めました。では、説明にはどの式を使えばよいでしょうか。対数の式の係数は「1単位増えると価格が何%変わるか」という読み方になり、少しややこしくなります。そこでここからは、係数がドルでそのまま読める、そのままの式で説明を考えます。当てる式と説明する式を使い分けるのも、実務ではよくあることです。

線形回帰の係数`m.coef_`は、それぞれの列が1単位増えたときに、予測価格が何ドル変わるかを表します。


In [ ]:
coef = pd.Series(m.coef_, index=feat)
print(coef.round(1))


係数を、お客さんに伝わる日本語の文に直します。数を文の中に埋め込むのに便利なのがf-stringです。


### お手本：係数を文にする

- 役割：居住面積の係数を、日本語の文の中に埋め込んで表示する
- 入力：係数の並び`coef`
- 出力：1行の文

文字列の前に`f`を付けると、`{ }`の中に書いた式の値が、文の中に埋め込まれます。`{値:,.0f}`の`:,.0f`は「3けたごとにカンマを入れ、小数点以下0けた」という書式です。


In [ ]:
area = coef['Gr Liv Area']
print(f'居住面積が1㎡広いと、ほかの条件をそろえたうえで、'
      f'価格は約{area:,.0f}ドル高い')


居住面積は1㎡あたり約677ドルです。

重回帰（multiple regression）の係数は、「ほかの列の値を変えずに、その列だけを1単位動かしたら」の変化です。677ドルは、品質も、建てた年も、部屋の数も同じ家どうしで比べたときの差です。この一言を落とすと、「広い家は品質も新しさも違うから高い」というほかの条件の影響まで、面積のおかげだと読んでしまいます。


### 変えてみる：品質の係数も文にする


In [ ]:
qual = coef['Overall Qual']
print(f'品質が1段階高いと、ほかの条件をそろえたうえで、'
      f'価格は約{qual:,.0f}ドル高い')


品質は1段階あたり約15,968ドルです。


---


## 6. おかしな符号と、大きさの比べ方


5節では、面積と品質の係数を文にしました。ほかの列の係数も、同じようにそのまま読めるでしょうか。係数を見ると、バスルームの数（`Full Bath`）が−8,610ドル、部屋の数（`TotRms AbvGrd`）が−2,911ドル、車庫の台数（`Garage Cars`）が−312ドルと、マイナスになっています。お風呂が1つ多いと、家は安くなるのでしょうか？


### お手本：バスルームの数だけの式と比べる

- 役割：バスルームの数だけで式を立て、11本の式の係数と比べる
- 入力：訓練データ
- 出力：2つの係数と、関係する列の相関


In [ ]:
m_bath = LinearRegression().fit(tr[['Full Bath']], y_tr)
print(f"バスルームだけの式: 1つ多いと {m_bath.coef_[0]:+,.0f} ドル")
print(f"11本の列の式      : 1つ多いと {coef['Full Bath']:+,.0f} ドル")
cols3 = ['Full Bath', 'Gr Liv Area', 'TotRms AbvGrd']
print(tr[cols3].corr().round(2))


バスルームの数だけの式では、1つ多いと+71,941ドル。ところが11本の式では−8,610ドルです（`:+,.0f`の`+`は、プラスのときも符号を付ける書式です）。

理由は「ほかの条件をそろえたうえで」にあります。バスルームが多い家は、たいてい広い家です（居住面積との相関は0.65）。11本の式では、広さの影響は面積の係数が受け持っています。すると、バスルームの係数が表すのは「同じ広さなら、バスルームが1つ多い家」の差になります。そういう家は、ほかの部屋がそのぶん狭くなりがちです。マイナスは、その差を表していると考えられます。係数の符号は、どの列を一緒に入れたかで変わるのです。

では、価格にいちばん大きく関わっているのはどの列でしょうか。品質の15,968ドルは「1段階」あたり、面積の677ドルは「1㎡」あたりで、単位が違うのでそのまま比べられません。そこで、第3回の標準化を使います。


### 変えてみる：標準化してから式を立て、係数を並べる

各列を「平均0・ばらつき1」にそろえてから式を立てると、係数は「その列が1標準偏差（standard deviation）動いたら、価格が何ドル変わるか」になり、大きさを比べられます。`sort_values(key=abs, ascending=False)`は、絶対値の大きい順に並べる書き方です。


In [ ]:
sc = StandardScaler().fit(X_tr)                    # 物差しは訓練で
m_std = LinearRegression().fit(sc.transform(X_tr), y_tr)
coef_std = pd.Series(m_std.coef_, index=feat)
print(coef_std.sort_values(key=abs, ascending=False).round(0))


いちばん大きいのは居住面積（29,984ドル）、次が品質（21,338ドル）、3番目が地下室の面積（13,379ドル）です。1㎡あたりでは小さく見えた面積が、「1標準偏差（約45㎡）あたり」では最大になりました。単位をそろえて初めて、大きさが比べられるのです。

最後に、車庫の台数（`Garage Cars`）が−227ドルと、ほぼゼロなのはなぜでしょうか。車庫の台数と車庫の面積は、ほとんど同じことを表しているからです。


In [ ]:
print(tr['Garage Cars'].corr(tr['Garage Area']).round(2))
feat_no_area = [f for f in feat if f != 'Garage Area']   # 面積を外す
m_no_area = LinearRegression().fit(tr[feat_no_area], y_tr)
cars = m_no_area.coef_[feat_no_area.index('Garage Cars')]
print(f'車庫の面積を外すと、台数の係数は {cars:+,.0f} ドル')
print(mae(y_te, m_no_area.predict(te[feat_no_area])))


相関は0.88。車庫の面積も入れた式では台数の係数は−312ドルでしたが、面積を外すと+7,725ドルに変わりました。それなのに、外れ幅は19,061ドルから19,373ドルへ、ほとんど変わりません。

車庫の台数と面積は、ほぼ同じ情報を持っています。その情報を台数の係数で受け持っても、面積の係数で受け持っても、当たり具合はほとんど同じです。そのため二つへの分け前がうまく決まらず、係数が揺れます。これを多重共線性（multicollinearity）と呼びます。似た列の組は、係数を一つずつ読まず、合わせて読みます。


---


## 7. この家は、なぜこの値段なのか（査定書を作る）


6節までは、列ごとに係数を読んできました。最後に、1軒の家で式を読みます。1軒の家について「平均的な家より、なぜ高い（安い）のか」を、条件ごとに分けて説明します。線形回帰では、予測価格は次のように分けられます。

予測価格 ＝ 訓練の家の平均価格 ＋ Σ（係数 ×（その家の値 − 訓練の平均の値））

「係数 ×（その家の値 − 平均）」が、その条件による押し上げ（または押し下げ）です。

この分け方ができるのは、係数がドルで足し算になる、そのままの式`m`です。ですから査定書の予測も押し上げも、すべて説明用のそのままの式のものです（4節で選んだ当てる式＝対数の式の予測とは、少しずれます）。


### お手本：テストの最初の家の査定書

- 役割：1軒の家の、条件ごとの押し上げ・押し下げを計算し、査定書の文にする
- 入力：家1軒`house`、係数`coef`、訓練の平均
- 出力：押し上げの大きい順の並びと、査定書の文


In [ ]:
house = te.iloc[[0]]
contrib = (house[feat].iloc[0] - X_tr.mean()) * coef   # 押し上げ
contrib = contrib.sort_values(key=abs, ascending=False)
print(contrib.round(0).head(4))
pred = m.predict(house[feat])[0]
print(f'予測 {pred:,.0f} ドル（平均的な家 {y_tr.mean():,.0f} ドル）')
print(f'理由: {contrib.index[0]} で {contrib.iloc[0]:+,.0f} ドル、'
      f'{contrib.index[1]} で {contrib.iloc[1]:+,.0f} ドル')


この家の予測は108,700ドル（実際は131,000ドル）で、平均的な家（175,807ドル）より約6万7千ドル安い予測です。その内訳は、居住面積が狭いこと（約94㎡）で−29,728ドル、品質が5段階であることで−16,158ドル、1956年から改修していないことで−9,353ドル……と読めます。

査定担当としての答えは「この家が平均より安いのは、主に、狭さ（約3万ドル）と品質（約1万6千ドル）のため」。ただし、車庫の台数と面積のような似た列の組は、一つずつの押し上げを読まず、合わせて伝えます。


### TODO④：あなたの番。別の家の査定書を作る

テストデータから別の家（例えば`te.iloc[[5]]`）を選んで、お手本と同じように査定書の文を表示してください。


In [ ]:
# TODO: 別の家を選び、お手本と同じく押し上げを計算して、

# 解答例④：テストの6番目の家
house = te.iloc[[5]]
contrib = (house[feat].iloc[0] - X_tr.mean()) * coef
contrib = contrib.sort_values(key=abs, ascending=False)
print(contrib.round(0).head(4))
pred = m.predict(house[feat])[0]
print(f'予測 {pred:,.0f} ドル（平均的な家 {y_tr.mean():,.0f} ドル）')
print(f'理由: {contrib.index[0]} で {contrib.iloc[0]:+,.0f} ドル、'
      f'{contrib.index[1]} で {contrib.iloc[1]:+,.0f} ドル')


まだ言えないこと

- 似た列の組の扱い方：車庫の台数と面積のように揺れる係数を、どう落ち着かせるか
- 80本の列をどう選ぶか：今日は11本を選んだが、残りの列を入れると係数はどうなるか
- 当てる式と説明する式：対数の式の係数を、どう読めばよいか


---


## 目標に答えられたか


- 今日の目標は、式を選ぶ物差しを決め、「この家は、なぜこの値段なのか」を説明することでした。4節で、どちらの式を選びましたか？ その根拠は何でしたか？
- TODO②で、MAEとRMSEで式の順位が入れ替わったのはなぜですか？TODO③の交差検証では、どうなりましたか？
- バスルームの数の係数がマイナスになったのはなぜですか？ 「お風呂が多いと安い」と伝えてよいでしょうか？
- 7節の家が平均より安いのは、主にどの条件のためでしたか？


---


## 今日の要点


- MAEは外れの大きさの平均、RMSEは二乗して平均しルートを取った値。大外れが混ざるとRMSEが大きくなる
- 右に長く歪んだ価格は、対数を当ててから戻すと、大外れもマイナスの価格も減らせる
- 物差しを変えると、式の順位が入れ替わることがある。物差しを決め、交差検証の平均で比べてから選ぶ
- 係数は「ほかの条件をそろえたうえで」の変化。単独の関係とは符号まで変わることがある
- 単位の違う係数は、標準化してから比べる。似た列の組の係数は、合わせて読む
- Python：関数（`mae`・`rmse`）を`for`文の中で呼び、結果を辞書にためて表にする。f-stringで数を文に埋め込む


---


## 次回へ


今日は、11本の列で式を立て、係数を説明に変えました。でも、車庫の台数と面積のような似た列の組があると、係数は揺れてしまいました。

次回は、列を35本に増やします。当たりは良くなるのに、係数はもっとおかしくなります。次回の問いは、列を増やして当たりを良くしながら、お客さんに説明できる係数を保てるか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

6節では、車庫の面積を外して式を立て直しました。今度は逆に、車庫の台数（`Garage Cars`）を外して、残りの10本の列で線形回帰を立ててください。そして、車庫の面積（`Garage Area`）の係数と、テストでの外れ幅（`mae`）を表示してください。


In [ ]:
feat_no_cars = [f for f in feat if f != 'Garage Cars']
m_no_cars = LinearRegression().fit(tr[feat_no_cars], y_tr)
area_coef = m_no_cars.coef_[feat_no_cars.index('Garage Area')]
print(f'台数を外した式: 車庫の面積の係数 {area_coef:+,.1f} ドル')
print('MAE:', mae(y_te, m_no_cars.predict(te[feat_no_cars])), 'ドル')


<details><summary>詰まったら</summary>

6節の最後のセルの`f != 'Garage Area'`を`f != 'Garage Cars'`に変えます。変数の名前は`feat_no_cars`、`m_no_cars`のように別の名前にしておくと、6節の結果と比べやすくなります。係数は`m_no_cars.coef_[feat_no_cars.index('Garage Area')]`で取り出せます。

</details>


### 応用②（判断）

応用①で、車庫の台数を外したときの、車庫の面積（`Garage Area`）の係数は何ドルですか。四捨五入して整数（ドル）で答えてください（例: 123）。


In [ ]:
print('答え:', round(area_coef))


<details><summary>詰まったら</summary>

`round(係数)`で整数にします。11本の列を入れた式での車庫の面積の係数（5節の`coef['Garage Area']`、約369ドル）と比べておきましょう。

</details>


### 応用③（解釈）

6節（車庫の面積を外す）と応用①（車庫の台数を外す）の結果をふまえて、「車庫は、この家の値段にどれくらい影響しているか」を、家を売りたいお客さんに向けて3行程度で説明してください。


（模範例）

車庫の台数と車庫の面積はほぼ同じことを表していて（相関0.88）、両方を式に入れると、台数の係数は−312ドルと、まるで車庫が値段を下げるかのように見えてしまいます。

ところが、面積を外すと台数1台あたり約7,700ドル、台数を外すと面積1㎡あたり約360ドル（1台分の約25㎡で約9千ドル）と、どちらで見ても車庫は値段を押し上げていて、外れ幅もほとんど変わりません。

ですから「車庫は値段に関係ない」のではなく、「車庫は、1台分の広さでおよそ8千〜9千ドル、値段を押し上げている」と伝えるのが正確です。


---


## 発展（任意）


### 係数は、どれだけ揺れるのか

6節で、似た列の組があると係数の分け前が決まらず揺れる、と書きました。本当にそうか、訓練データから重複を許して同じ数だけ取り出す（ブートストラップ）ことを100回くり返し、そのたびに式を立てて、車庫の台数の係数がどれだけ揺れるかを見ます。


In [ ]:
gc_with, gc_without = [], []
for i in range(100):
    s = tr.sample(len(tr), replace=True, random_state=i)   # 重複を許して取り出す
    c1 = LinearRegression().fit(s[feat], s['SalePrice']).coef_
    c2 = LinearRegression().fit(s[feat_no_area], s['SalePrice']).coef_
    gc_with.append(c1[feat.index('Garage Cars')])
    gc_without.append(c2[feat_no_area.index('Garage Cars')])
for label, v in [('面積も入れた式', gc_with), ('面積を外した式', gc_without)]:
    lo, hi = np.percentile(v, [2.5, 97.5])
    neg = np.mean(np.array(v) < 0) * 100
    print(f'{label}: 95%が {lo:+,.0f} 〜 {hi:+,.0f} ドル, マイナスの回 {neg:.0f} %')


読み方　車庫の面積も入れた式では、車庫の台数の係数は、100回のうち95回が−3,937〜+3,205ドルの幅に入り、64%の回でマイナスでした。符号さえ定まらないのです。車庫の面積を外すと、係数は+5,262〜+9,642ドルの幅に収まり、一度もマイナスになりません。

同じデータから少し違う訓練データを取っただけで、似た列の組の係数は、プラスにもマイナスにも振れます。これが、多重共線性で「係数を一つずつ読んではいけない」理由です。

試すなら、居住面積（`Gr Liv Area`）と部屋の数（`TotRms AbvGrd`、相関0.81）の組でも、部屋の数の係数がどれだけ揺れるかを見てみましょう。
